---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-41: Agentic AI and Tool Calling - II</h1>

# Learning agenda of this notebook
- Recap: Workflow of a Tool Call

1. **Example 1:** Calling a Custom Function **(Multiply)** using OpenAI's `Responses` API using Groq Hosted Model
    - Step 1: Define the Function and Tool Schema
    - Step 2: Ask the Model (Initial Request)
    - Step 3: Make the Tool call (if Required) and Send the tool result to Model
    - Step 4: Display the final result
2. **Example 2:** Calling a Custom Function **(Multiply)** using OpenAI's `Chat Completions` API using Groq Hosted Model
    - Step 1: Define the Function and Tool Schema
    - Step 2: Ask the Model (Initial Request)
    - Step 3: Make the Tool call (if Required) and Send the tool result to Model
    - Step 4: Display the final result
3. **Example 3:** Calling a Custom Function **(Get Current Date/Time)** Using OpenAI's `Chat Completions` API using Groq Hosted Model
4. **Example 4:** Calling a Custom Function **(Get Weather via API)** Using OpenAI's `Chat Completions` API using Groq Hosted Model
5. **Example 5:** A ReAct Agent with Native Tool Calling (Agent = Model + Harness)
    - From Lec-39's ReAct to native tool calling
    - The ReAct loop, and Agent = Model + Harness
    - A ReAct agent with this lecture's tools, printing every Think → Act → Observe step
    - The same model without a harness

> **Note:** The Groq examples use `openai/gpt-oss-20b`, except Example 5, which uses `openai/gpt-oss-120b` (its markdown cell explains why). Groq retired `llama-3.3-70b-versatile` and `llama-3.1-8b-instant` on August 16, 2026 (replacements: `openai/gpt-oss-120b` and `openai/gpt-oss-20b`). Check https://console.groq.com/docs/deprecations if a model returns a `404 model_not_found` error.


# <span style='background :lightgreen' >Recap: Workflow of a Tool Call</span>

<div style="text-align:center;">
    <img src="../images/tool-call-workflow.svg"
         style="max-width:1400px; width:100%; height:auto; display:inline-block;">
</div>

# <span style='background :lightgreen' >1. Example 1: Calling a Custom Function (Multiply) using OpenAI's `Responses` API using Groq Hosted Model</span>

## Step 1: Define the Function and Tool Schema (Tool Specification)
- You define the actual Python function that runs on your machine when the model calls your tool. Type hints and a docstring make the function easier to read, but **the model never sees them**: with the raw API, the model sees only the tool schema (Lec-39, Section 3 (vii)). Frameworks such as FastMCP (Lec-45) and the OpenAI Agents SDK (Lec-50) build the schema *from* the type hints and docstring, so there they do matter.
- Tool Schema (JSON) describes your tool to the model - what it does, what parameters it accepts, and when to use it. This is what the model "sees" and uses to decide whether to call your tool. 
- Create a flattened JSON schema for the Responses API with four essential elements:
    - **type:** Specifies that this tool represents a callable function. In this code → "function". For user-provided tools, type="function" is the only supported option.
    - **name:** A short, descriptive function identifier. In this code → "multiply"
    - **description:** A clear explanation of what the function does and when to use it. In this code → "Multiply two numbers"
    - **parameters:** A JSON Schema object describing the input fields, their types, and any required arguments.
- This schema tells the LLM what functions are available and how to call them
- Key difference from Chat Completions: No nested "function" wrapper object
- **Best practice:** Write the `description` for the model, not for humans. Say *when* the tool should (and should not) be used. Optionally add `"strict": True` together with `"additionalProperties": False` so the arguments are guaranteed to match your schema.

In [1]:
# ================================================================================================================================
# Define the Python function that will execute on local machine
# Type hints and a docstring document the function for humans. The model never sees them: it sees only the tool schema below
# =================================================================================================================================
def multiply(a: float, b: float) -> float:
    """Multiply two numbers"""
    return a * b

# ===============================================================================================================================================================================================
# Define the tool schema in JSON. The following schema tells the model: "You may call a function named multiply, which requires a JSON object with two numeric fields (a and b), both mandatory."
# ===============================================================================================================================================================================================
tools = [{
    "type": "function",           # "type" defines the kind of tool being declared. For your own (custom) functions it is always "function"; built-in tools use other types (web_search, file_search, code_interpreter, ...)
    "name": "multiply",           # Public unique identifier of the tool/function. The LLM mentions this name when calling the tool (inside tool_calls). Must match the actual function name you will execute locally.
    "description": "Multiply two numbers", # "description" is human readable explanation of what the tool does that helps both developers and LLMs understand: when to call tool, what the tool achieves etc
    "parameters": {               # "parameters" field describes what arguments the model must provide when calling your tool.
        "type": "object",         # The arguments always arrive as one JSON object (key-value pairs), so the top-level type of "parameters" is always "object". Each property inside it can be a string, number, integer, boolean, array or object.
        "properties": {           # "properties" defines the allowed and expected fields inside the object. Each property has its own JSON schema (Here we have two parameters "a" and "b")
            "a": {"type": "number"},
            "b": {"type": "number"}
        },
        "required": ["a", "b"]    # "required" specifies which parameters are mandatory. Missing a required field triggers immediate validation failure
    }
}]
print("=== STEP 1: Tool Defined (Responses API) ===")
print(f"Available tool: {tools[0]['name']}") 

=== STEP 1: Tool Defined (Responses API) ===
Available tool: multiply


## Step 2: Ask the Model (Initial Request)
- Now you send your initial request to the model, specifying the `tools` parameter as the tool schema you created and the `tool_choice` parameter as auto
- Send your user query using `client.responses.create()` with:
    - input=user_query (string format, not messages array)
    - tools=tools (your function definitions)
    - tool_choice="auto" (let model decide when to call functions)
- The model analyzes the request and determines if it needs to call a function
- Parse the response by iterating through response.output looking for items with type="function_call"
- Extract function name from item.name and arguments from json.loads(item.arguments)
- Key difference: Look for "function_call" type, not "tool_call"

In [2]:
import os                                   # Provides access to environment variables and operating system utilities.
import json                                 # Encodes and decodes data in JSON format.
from dotenv import load_dotenv              # Loads environment variables from a .env file into the runtime.
from openai import OpenAI                   # Official OpenAI client for sending requests to OpenAI models and receiving responses.

load_dotenv('../keys/.env', override=True) # Opens the .env file in the pwd or the specified file path, reads key=value pairs and insert them into os.environ and returns True/False depending the file exist or not
groq_api_key = os.getenv("GROQ_API_KEY")
groq_client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_api_key) # This client lets you use Groq-hosted models (like gpt-oss-20b) as if they were OpenAI models.

print("\n=== STEP 2: Initial Request to the Model ===")
#user_query = "What is the highest mountain peak in Pakistan?"   # → tool NOT needed
user_query = "What is 6 times 7?"                                # → tool WILL be used

# Send the initial request to the model with available tools (only one in this case)
response = groq_client.responses.create(
    model="openai/gpt-oss-20b",  # Other Groq models with tool calling: openai/gpt-oss-120b, qwen/qwen3.8-27b
    input=user_query,  
    tools=tools,
    tool_choice="auto" # auto, required, none
)

# =============================================================================================================
# Iterate through the response.output and check `item.type=="function_call"` to check if a tool call is required
# =============================================================================================================
tool_call = None
args = None
for item in response.output:
    if item.type == "function_call":                # Model requested function # Check for 'function_call' type (not 'tool_call')
        tool_call = item
        print(f"Tool call: {tool_call}")
        print(f"Tool requested: {tool_call.name}")
        args = json.loads(tool_call.arguments)      # Parse the arguments from the function call
        print(f"Arguments parsed: {args}")
        break


=== STEP 2: Initial Request to the Model ===
Tool call: ResponseFunctionToolCall(arguments='{"a":6,"b":7}', call_id='fc_26651ed1-a02e-4d04-8d61-1955520b7d89', name='multiply', type='function_call', id='fc_26651ed1-a02e-4d04-8d61-1955520b7d89', namespace=None, status='completed')
Tool requested: multiply
Arguments parsed: {'a': 6, 'b': 7}


## Step 3: Make the Tool call (if Required) and Send the tool result to Model
- If a tool call is made, you need to execute the function with appropriate arguments and save the result in, say, a `tool_result` variable.
- Then call `client.responses.create()` again with an `input` list containing:
    - the original user message,
    - the `function_call` item the model returned (`response.output`), and
    - a `{"type": "function_call_output", "call_id": ..., "output": ...}` item carrying your tool's result. The `call_id` links the result to the call.
- (With OpenAI you could instead pass `previous_response_id` and send only the `function_call_output`. Groq's Responses API is stateless, so we resend the history ourselves.)
- If a tool call is NOT made, simply display the initial response.
- Example 2 does the same with the `Chat Completions` API, where the result goes back in structured role messages instead: `user`, `assistant` with `tool_calls`, and `tool`.


In [3]:
print("\n=== STEP 3: Check if the Tool call is Required and make second call if needed ===")
final_response = None  # Variable to store the final response
# Check if the model wants to call a tool
if tool_call:
    print("Executing the function")
    tool_result = multiply(args["a"], args["b"])
    # Make second API call with the tool result
    print("Making the second call, and sending result of tool call for formatting")
    # ================================================================
    # Send tool result back to model - RESPONSES API FORMAT
    # ================================================================
    input_items = [
        {"role": "user", "content": user_query},          # 1. The original user message
        tool_call.model_dump(exclude_none=True),           # 2. The function_call item exactly as the model returned it
        {                                                  # 3. The result of running the function locally
            "type": "function_call_output",
            "call_id": tool_call.call_id,                  # Links this result to the function call above
            "output": str(tool_result)                     # Output must be a string (use json.dumps() for dicts)
        }
    ]

    final_response = groq_client.responses.create(
            model="openai/gpt-oss-20b",  # Other Groq models with tool calling: openai/gpt-oss-120b, qwen/qwen3.8-27b
            input=input_items,
            tools=tools
        )
else:
    print("No tool call needed. Using direct response.")
    final_response = response



=== STEP 3: Check if the Tool call is Required and make second call if needed ===
Executing the function
Making the second call, and sending result of tool call for formatting


## Step 4: Display the final result

In [4]:
print("\n=== STEP 4: Final Result ===")
if tool_call:
    print("Result from second API call (after tool execution):")
else:
    print("Result from first API call (direct response):")
print(final_response.output_text)


=== STEP 4: Final Result ===
Result from second API call (after tool execution):
6 times 7 equals **42**.


# <span style='background :lightgreen' >2. Example 2: Calling a Custom Function (Multiply) using OpenAI's `Chat Completions` API using Groq Hosted Model</span>

## Step 1: Define the Function and Tool Schema

In [5]:
# ================================================================================================================================
# Define the Python function that will execute on local machine
# Type hints and a docstring document the function for humans. The model never sees them: it sees only the tool schema below
# =================================================================================================================================
def multiply(a: float, b: float) -> float:
    """Multiply two numbers"""
    return a * b

# ================================================================
# Define tool schema (inside 'function' object)
# ================================================================
tools = [{
    "type": "function",
    "function": {
        "name": "multiply",
        "description": "Multiply two numbers",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {"type": "number"},
                "b": {"type": "number"}
            },
            "required": ["a", "b"]
        }
    }
}]

print("=== STEP 1: Tool Defined (Chat Completions API) ===")
print(f"Available tool: {tools[0]['function']['name']}")

=== STEP 1: Tool Defined (Chat Completions API) ===
Available tool: multiply


## Step 2: Ask the Model (Initial Request)

In [8]:
import os                                   # Provides access to environment variables and operating system utilities.
import json                                 # Encodes and decodes data in JSON format.
from dotenv import load_dotenv              # Loads environment variables from a .env file into the runtime.
from openai import OpenAI                   # Official OpenAI client for sending requests to OpenAI models and receiving responses.

load_dotenv('../keys/.env', override=True) # Opens the .env file in the pwd or the specified file path, reads key=value pairs and insert them into os.environ and returns True/False depending the file exist or not
groq_api_key = os.getenv("GROQ_API_KEY")
groq_client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_api_key) # This client lets you use Groq-hosted models (like gpt-oss-20b) as if they were OpenAI models.


print("\n=== STEP 2: Initial Request to the Model ===")
#user_query = "What is the highest mountain peak in Pakistan?"   # → tool NOT needed
user_query = "What is 6 times 7?"                                # → tool WILL be used

# Send the initial request to the model with available tools (only one in this case)
response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",  # Other Groq models with tool calling: openai/gpt-oss-120b, qwen/qwen3.8-27b
    messages=[{"role": "user", "content": user_query}],
    tools=tools,
    tool_choice="auto"
)
# Check if the model wants to call a tool
if response.choices[0].message.tool_calls:
    print("Tool call detected.")
else:
    print("No tool call needed.")


=== STEP 2: Initial Request to the Model ===
Tool call detected.


## Step 3: Make the Tool call (if Required) and Send the tool result to Model

In [10]:
print("\n=== STEP 3: Check if the Tool call is Required and make second call if needed ===")
final_response = None  # Variable to store the final response

# Check if the model wants to call a tool
if response.choices[0].message.tool_calls:
    # Extract tool call details
    tool_call = response.choices[0].message.tool_calls[0]
    function_name = tool_call.function.name
    function_args = json.loads(tool_call.function.arguments)
    
    # Execute the Python function (in this case, multiply)
    if function_name == "multiply":
        print("Executing the function")
        tool_result = multiply(function_args["a"], function_args["b"])
    else:
        tool_result = f"Error: unknown tool {function_name}"   # The model asked for a tool we do not have: report it instead of crashing

    # Make second API call with the tool result
    print("Making the second call, and sending result of tool call for formatting")
    # ===============================================================
    # Generate the messages with user query, assistant's tool call and tool's result
    # ===============================================================
    messages = [
        {"role": "user", "content": user_query},
        {"role": "assistant", "content": None,  # Assistant gave no direct text, only tool call
                                            "tool_calls": [{
                                                            "id": tool_call.id,
                                                            "type": "function",
                                                            "function": {"name": tool_call.function.name, "arguments": tool_call.function.arguments}
                                                            }]
        },
        {"role": "tool", "tool_call_id": tool_call.id, "content": str(tool_result)}
    ]

    final_response = groq_client.chat.completions.create(
            model="openai/gpt-oss-20b",  # Other Groq models with tool calling: openai/gpt-oss-120b, qwen/qwen3.8-27b
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
else:
    print("No tool call needed. Using direct response.")
    final_response = response


=== STEP 3: Check if the Tool call is Required and make second call if needed ===
Executing the function
Making the second call, and sending result of tool call for formatting


## Step 4: Display the final result

In [11]:
print("\n=== STEP 4: Final Result ===")
if response.choices[0].message.tool_calls:
    print("Result from second API call (after tool execution):")
else:
    print("Result from first API call (direct response):")

print(final_response.choices[0].message.content)


=== STEP 4: Final Result ===
Result from second API call (after tool execution):
42


# <span style='background :lightgreen' >3. Example 3: Calling a Custom Function (Get Current Date/Time) Using OpenAI's `Chat Completions` API using Groq Hosted Model</span>

- Examples 1 and 2 made exactly **two** LLM calls: one to get the tool call, and one to turn the result into an answer. That fixed flow breaks as soon as the model needs **more than one tool call**, for example *"What time is it in Lahore and in New York?"*, or a question whose second step depends on the first result.
- From this example on, the code has **five parts**, and you will see the same five parts again in Example 4, Example 5 and Lec-42:
    1. **The tool:** a plain Python function, `get_current_time(timezone)`, which uses the standard-library `zoneinfo` module.
    2. **`TOOLS` and `TOOL_SCHEMAS`:** `TOOLS` maps each tool name to its function (a *dispatch table*), and `TOOL_SCHEMAS` describes each tool to the model. Adding a tool means adding one entry to each.
    3. **`run_tool()`:** looks up the function by name and calls it with the model's arguments (`**args`). If anything fails (an unknown timezone such as `Mars/Base`, or a tool name the model made up), it returns the error **as text**: the model reads it as the tool result, and the program does not crash.
    4. **`run_agent()`, the agent loop:** call the model; if it asks for tools, run each one (the model may ask for several in one reply: *parallel tool calls*), append each result as a `"tool"` message, and call the model again. The loop ends when the model replies with plain text.
    5. **`main()`, the chat loop:** keeps the whole conversation in `messages`, so the agent remembers earlier turns (Lec-39, Example 3).
- The system prompt tells the model **when** to use the tool, and to answer other questions directly.

**Questions to ask** (type each one at the `You:` prompt, then type `exit` to finish the cell):
1. *Hello, I am Arif.* and then *Tell me my name.* (no tool: the answer comes from `messages`)
2. *What is the color of the sky?* (no tool)
3. *Tell me the current date and time.* (one tool call)
4. *What time is it in Lahore and in New York?* (two tool calls)

**What to notice:** a `tool: ...` line is printed only when the model asks for a tool, so you can see which questions needed one. The time in the answer comes from your machine's clock, not from the model's guess.

In [1]:
import os
import json
from datetime import datetime
from zoneinfo import ZoneInfo                # Standard-library IANA timezone support (Python 3.9+)
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv('../keys/.env', override=True)
groq_api_key = os.getenv("GROQ_API_KEY")

client = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")


MODEL = "openai/gpt-oss-20b"                 # Other Groq models with tool calling: openai/gpt-oss-120b, qwen/qwen3.8-27b

SYSTEM_PROMPT = """You are a helpful assistant.
When the user asks for the current date or time anywhere in the world, use the get_current_time tool.
Answer other questions directly."""


# ================================================================
# 1. The tool: a plain Python function that runs on your machine
# ================================================================
def get_current_time(timezone):
    """Return the current date and time in an IANA timezone, e.g. 'Asia/Karachi'."""
    now = datetime.now(ZoneInfo(timezone))
    return now.strftime("%Y-%m-%d %H:%M:%S %Z")


# ================================================================
# 2. Map each tool name to its function, and describe each tool to the model
# ================================================================
TOOLS = {
    "get_current_time": get_current_time,
}

TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Get the current date and time for a given timezone.",
            "parameters": {
                "type": "object",
                "properties": {
                    "timezone": {"type": "string", "description": "IANA timezone name, e.g. Asia/Karachi, America/New_York"},
                },
                "required": ["timezone"],
            },
        },
    },
]


# ================================================================
# 3. Run one tool call: look up the function by name and call it with the model's arguments
# ================================================================
def run_tool(tool_call):
    name = tool_call.function.name                    # Name of the function the model wants to call, e.g. "get_current_time"
    args = json.loads(tool_call.function.arguments)   # Arguments arrive as a JSON string; parse them into a dict, e.g. {"timezone": "Asia/Karachi"}
    print(f"  tool: {name}({args})")                  # Show which tool is being run, and with which arguments
    try:                                              # A failing tool must not crash the agent
        return str(TOOLS[name](**args))               # Look up the function by name, call it with the arguments as keyword arguments (**args), return the result as a string
    except Exception as error:                        # e.g. an unknown timezone such as "Mars/Base", or a tool name the model made up
        return f"Error: {error}"                      # Return the error as text: the model reads it as the tool result and can try again


# ================================================================
# 4. The agent loop: call the model, run any tools it asks for, repeat until it answers in plain text
# ================================================================
def run_agent(messages):
    while True:                                       # Keep looping until the model answers without asking for a tool
        response = client.chat.completions.create(    # Send the whole conversation to the model:
            model=MODEL,                              #   the model to use
            messages=messages,                        #   every message so far (system, user, assistant and tool messages)
            tools=TOOL_SCHEMAS,                       #   the tools the model is allowed to call
        )
        message = response.choices[0].message         # The model's reply: either plain text, or one or more tool calls
        messages.append(message)                      # Add the reply to the conversation (the API needs the tool-call message before its results)

        # No tool calls means the model is done and answered in plain text
        if not message.tool_calls:                    # The model did not ask for any tool...
            return message.content                    # ...so its text is the final answer for the user

        for tool_call in message.tool_calls:          # The model may ask for several tools in one reply (parallel tool calls)
            result = run_tool(tool_call)              # Run the requested function on this machine
            messages.append({                         # Add the result to the conversation as a "tool" message:
                "role": "tool",                       #   marks this message as a tool result
                "tool_call_id": tool_call.id,         #   links the result to the tool call it answers
                "content": result,                    #   the tool's output (always a string)
            })
        # Back to the top of the while loop: the model now sees the tool results and decides what to do next


# ================================================================
# 5. The chat loop: keep the conversation in `messages` and answer until the user types exit
# ================================================================
def main():
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    print("Time assistant ready. Type 'exit' to quit.")
    while True:
        user_input = input("\nYou: ")
        if user_input.strip().lower() in ("exit", "quit"):
            break
        messages.append({"role": "user", "content": user_input})
        reply = run_agent(messages)
        print(f"\nAgent: {reply}")


if __name__ == "__main__":
    main()

Time assistant ready. Type 'exit' to quit.



You:  hello i am arif



Agent: Hello Arif! How can I help you today?



You:  tell me my name



Agent: Your name is Arif.



You:  what is the color of sky



Agent: During most of the day, the sky looks **blue**. That happens because air molecules scatter shorter (blue) wavelengths of sunlight more efficiently than longer wavelengths, so the light that reaches our eyes from all directions is dominated by blue. In the early morning or late afternoon it can shift to pink, orange, or red, and at night it is dark (or star‑filled).



You:  tell me the current date and time


  tool: get_current_time({'timezone': 'UTC'})

Agent: Current date and time (UTC): **2026‑10‑21 07:12 UTC**



You:  exit


# <span style='background :lightgreen' >4. Example 4: Calling a Custom Function (Get Weather via API) Using OpenAI's `Chat Completions` API using Groq Hosted Model</span>

## Accessing Weather API from Python using their API key

In [13]:
# A simple run of calling the weatherapi.com
import requests
import os
from dotenv import load_dotenv

load_dotenv('../keys/.env', override=True) 
weather_api_key = os.getenv('WEATHER_API_KEY')
city='Lahore'

url = 'https://api.weatherapi.com/v1/current.json'
response = requests.get(url, params={'key': weather_api_key, 'q': city}, timeout=10)
response.raise_for_status()  # Raise an exception for bad status codes
data = response.json()
print(f"Country: {data['location']['country']}")
print(f"City: {data['location']['name']}")
print(f"Latitude: {data['location']['lat']}")
print(f"Longitude: {data['location']['lon']}")
print(f"Temperature: {data['current']['temp_c']}°C ({data['current']['temp_f']}°F)")
print(f"Condition: {data['current']['condition']['text']}")
print(f"Humidity: {data['current']['humidity']}%")
print(f"Wind: {data['current']['wind_kph']} km/h")
print(f"Feels like: {data['current']['feelslike_c']}°C")
print(f"Last updated: {data['current']['last_updated']}")

Country: Pakistan
City: Lahore
Latitude: 31.5497
Longitude: 74.3436
Temperature: 31.7°C (89.1°F)
Condition: Sunny
Humidity: 36%
Wind: 13.7 km/h
Feels like: 30.6°C
Last updated: 2026-10-10 10:15


## A weather agent: the same five parts, a different tool
- The tool `get_city_weather(city_name)` wraps the WeatherAPI.com call above. Two details matter for any tool that calls a web API:
    - `timeout=10`, so a slow server cannot hang the agent;
    - on failure it returns a short message, never the raw error, because the error text contains the request URL **with your API key** in it. Whatever a tool returns is sent to the LLM provider.
- `TOOLS`, `TOOL_SCHEMAS`, `run_tool()`, `run_agent()` and `main()` are the same as in Example 3. Only the tool changed.

**Questions to ask** (type each one at the `You:` prompt, then type `exit` to finish the cell):
1. *What's the current weather in Lahore?*
2. *Compare the weather in Karachi and Islamabad.*
3. *Which of these three cities is the warmest right now?*

**What to notice**
- **Question 2:** the model asked for both cities in a **single** reply (parallel tool calls), and `run_agent()` ran both before calling the model again.
- **Question 3:** in our run, the model answered from the results **already in `messages`** and made no new call, even though the system prompt says *"make a fresh call on every request"*. The numbers came from earlier in the conversation. A system prompt is a request, not a guarantee: if fresh data really matters, enforce it in code. Example 5 makes this kind of behaviour visible by printing every step.

In [14]:
import os
import json
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv('../keys/.env', override=True)
groq_api_key = os.getenv("GROQ_API_KEY")
weather_api_key = os.getenv("WEATHER_API_KEY")

client = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")


MODEL = "openai/gpt-oss-20b"                 # Other Groq models with tool calling: openai/gpt-oss-120b, qwen/qwen3.8-27b

SYSTEM_PROMPT = """You are a helpful weather assistant.
When the user asks about the weather in a city, use the get_city_weather tool, and make a fresh call on every request.
Answer other questions directly."""


# ================================================================
# 1. The tool: a Python function that calls the WeatherAPI.com web API
# ================================================================
def get_city_weather(city_name):
    """Return the current weather for a city, e.g. 'Lahore'."""
    url = "https://api.weatherapi.com/v1/current.json"                   # HTTPS, so the API key is not sent in plain text
    try:
        response = requests.get(url, params={"key": weather_api_key, "q": city_name.strip()}, timeout=10)
        response.raise_for_status()                                       # Raise an error for 4xx/5xx responses
        data = response.json()
        location, current = data["location"], data["current"]
    except (requests.RequestException, KeyError):
        return "Weather information not available for this city"         # Never return the raw error: its URL contains the API key
    return (f"Current weather in {location['name']}, {location['country']} "
            f"(Lat: {location['lat']}, Lon: {location['lon']}) as of {location['localtime']}: "
            f"{current['temp_c']}°C, {current['condition']['text']}, humidity {current['humidity']}%")


# ================================================================
# 2. Map each tool name to its function, and describe each tool to the model
# ================================================================
TOOLS = {
    "get_city_weather": get_city_weather,
}

TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "get_city_weather",
            "description": "Get the current weather for a city. Call this whenever the user asks what the weather is like in a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city_name": {"type": "string", "description": "The city to get the weather for, e.g. Lahore, Karachi"},
                },
                "required": ["city_name"],
            },
        },
    },
]


# ================================================================
# 3. Run one tool call: look up the function by name and call it with the model's arguments
# ================================================================
def run_tool(tool_call):
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)
    print(f"  tool: {name}({args})")
    try:
        return str(TOOLS[name](**args))
    except Exception as error:
        return f"Error: {error}"


# ================================================================
# 4. The agent loop: call the model, run any tools it asks for, repeat until it answers in plain text
# ================================================================
def run_agent(messages):
    while True:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOL_SCHEMAS,
        )
        message = response.choices[0].message
        messages.append(message)

        # No tool calls means the model is done and answered in plain text
        if not message.tool_calls:
            return message.content

        for tool_call in message.tool_calls:
            result = run_tool(tool_call)
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            })


# ================================================================
# 5. The chat loop: keep the conversation in `messages` and answer until the user types exit
# ================================================================
def main():
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    print("Weather assistant ready. Type 'exit' to quit.")
    while True:
        user_input = input("\nYou: ")
        if user_input.strip().lower() in ("exit", "quit"):
            break
        messages.append({"role": "user", "content": user_input})
        reply = run_agent(messages)
        print(f"\nAgent: {reply}")


if __name__ == "__main__":
    main()

Weather assistant ready. Type 'exit' to quit.

You: What's the current weather in Lahore?
  tool: get_city_weather({'city_name': 'Lahore'})

Agent: **Current weather in Lahore (Pakistan)**  
- **Temperature:** 33.0 °C  
- **Conditions:** Smoky haze  
- **Humidity:** 20 %  

If you need more details (wind speed, forecast, etc.) just let me know!

You: Compare the weather in Karachi and Islamabad.
  tool: get_city_weather({'city_name': 'Karachi'})
  tool: get_city_weather({'city_name': 'Islamabad'})

Agent: ### Weather comparison (October 21 2026, 12:15 local time)

| City | Temperature | Sky | Humidity | Other notes |
|------|-------------|-----|----------|-------------|
| **Karachi** | 28.8 °C | Sunny | 64 % | Coastal, moderate humidity. |
| **Islamabad** | 29.7 °C | Sunny | 22 % | Inland, dry and clear. |

**Key differences**

- **Temperature:** Islamabad is slightly warmer by about 0.9 °C.  
- **Humidity:** Karachi’s coastal location gives it high humidity (64 %) versus Islamabad’s l

# <span style='background :lightgreen' >5. Example 5: A ReAct Agent with Native Tool Calling (Agent = Model + Harness)</span>

<h3 align="center"><div class="alert alert-success" style="margin: 20px">In Lec-39 (Example 8) you built a ReAct agent from plain text and a regular expression. Examples 3 and 4 of this lecture ran the <b>same loop</b> with native tool calling, hidden inside <code>run_agent()</code>. This example opens that loop up and prints every <b>Think → Act → Observe</b> step.</div></h3>

### a. From Lec-39's ReAct to native tool calling
Each ReAct step still happens; only the way it travels between the model and our code has changed:

| ReAct step | Lec-39 Example 8 (text + regex) | Lec-41 Example 5 (native tool calling) |
|---|---|---|
| **Think** | A `Thought:` line written in the reply | `message.reasoning`: gpt-oss's hidden thinking, which Groq returns as a separate field |
| **Act** | `Action: lookup[LUMS]`, found by our regex | `message.tool_calls`: structured JSON with the tool's name and arguments |
| **Observe** | A user message: `"Observation: ..."` | A `"tool"` message, linked to its call by `tool_call_id` |
| **Final answer** | A reply with no `lookup[...]` in it | A reply with no `tool_calls` |
| **Guardrails** | `stop=["Observation:"]` and `range(1, 6)` | No `stop` needed: only our code can write a `"tool"` message, so the model cannot invent an observation. The step limit stays: `MAX_STEPS` |

### b. The ReAct loop

<div style="text-align:center;">
    <img src="../images/agent-new-03.png"
         style="max-width:1300px; width:100%; height:auto; display:inline-block;">
</div>

### c. Agent = Model + Harness

<div style="text-align:center;">
    <img src="../images/agent-new-06.png"
         style="max-width:1300px; width:100%; height:auto; display:inline-block;">
</div>

The model only **decides**: it reads the context and replies with either an answer or a tool call. Everything else is the **harness**, plain Python code around the model. In Example 5, every component in the figure maps to a few lines of code:

| Component (Lec-39, Section 3) | In the code of Example 5 | Model or harness? |
|---|---|---|
| Model | `MODEL = "openai/gpt-oss-120b"` | **Model**: one line |
| Instructions | `SYSTEM_PROMPT` | Harness |
| Tools | `multiply()`, `get_current_time()`, `get_city_weather()`, plus `TOOLS` and `TOOL_SCHEMAS` | Harness |
| Agent loop | `run_agent()` | Harness |
| Context and memory | `messages`, which keeps every question, tool call and observation | Harness |
| Guardrails | `MAX_STEPS`; tool errors returned as text instead of crashing | Harness |
| Observability | The printed `Thought:` / `Action:` / `Observation:` trace | Harness |

Section e below runs the **same model without the harness**, so you can see the difference for yourself.

### d. The code: a ReAct agent with this lecture's tools
- **Tools:** the three tools from Examples 2, 3 and 4 (`multiply`, `get_current_time` and `get_city_weather`). All three only *read* information, so nothing needs your approval, and everything runs on Groq's free tier. One small change: `get_city_weather()` now also returns the city's timezone (`tz_id`), which the model can pass to `get_current_time()`.
- **What changed from Examples 3 and 4:** `TOOLS`, `TOOL_SCHEMAS`, `run_tool()` and `main()` follow the same pattern. Only `run_agent()` changed, in two ways:
    - it **prints** each pass: the `Thought:` (shortened to 300 characters with `textwrap.shorten`), each `Action:` and each `Observation:`;
    - it uses `for step in range(1, MAX_STEPS + 1)` instead of `while True`, so it **stops safely** after `MAX_STEPS` passes.
- **Why `gpt-oss-120b`:** both `gpt-oss-20b` and `gpt-oss-120b` return their reasoning in `message.reasoning`, but the 20b model's Thoughts are much longer. The 120b model writes short Thoughts that are easy to follow. Changing the model is a one-line edit, and the harness stays the same.

**Questions to ask** (type each one at the `You:` prompt, then type `exit` to finish the cell):
1. *Which is warmer right now, Lahore or London? Then tell me the local time in the warmer city.*
2. *What is 1234.5 times 6789?*
3. *What is the capital of Pakistan?*
4. *What's the weather in Zxqwvbtown?*

**What to notice in the trace**
- Each `--- Pass N ---` block is **one LLM call**. The `Observation:` lines come from **our code**, not from the model.
- **Question 1** took four passes in our run: weather in Lahore → weather in London → time in Lahore → answer. Nobody wrote this plan. The model chose `get_current_time("Asia/Karachi")` only **after** reading both observations (21.8°C vs 7.4°C); had London been warmer, it would have asked for `Europe/London`. Some runs ask for both cities in a single pass (parallel tool calls) and finish in three passes.
- **Question 2** needs one tool call (two passes). **Question 3** needs none, so the loop ends on the first pass. A pass that gives the final answer often prints no `Thought:`, because the model went straight to the answer.
- **Question 4:** the tool fails, and its failure comes back as an observation: *"Weather information not available for this city"*. The model reports that honestly instead of inventing a temperature.
- **Try the step limit:** set `MAX_STEPS = 1` and ask question 1 again. After the first lookup, the loop runs out and the agent replies *"I reached the step limit (MAX_STEPS = 1) before finding an answer. Please try a simpler question."*
- `messages` lives for the whole session, as in Examples 3 and 4, so it grows with every Thought, Action and Observation, and follow-up questions can use earlier results.

In [ ]:
import os
import json
import requests
from datetime import datetime
from textwrap import shorten
from zoneinfo import ZoneInfo
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv('../keys/.env', override=True)
groq_api_key = os.getenv("GROQ_API_KEY")
weather_api_key = os.getenv("WEATHER_API_KEY")

client = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")


# ================================================================
# THE MODEL: the only line in this cell that is not harness
# ================================================================
MODEL = "openai/gpt-oss-120b"                # openai/gpt-oss-20b also works, but writes much longer Thoughts

# ================================================================
# THE HARNESS: everything below is plain code around the model
# ================================================================
MAX_STEPS = 5                                # Guardrail: at most MAX_STEPS passes (LLM calls) for each question

SYSTEM_PROMPT = """You are a helpful assistant with three tools: multiply, get_current_time and get_city_weather.
Use multiply for arithmetic, get_current_time for the current date or time, and get_city_weather for the weather in a city.
Answer other questions directly."""


# ================================================================
# 1. The tools: the same three functions as in Examples 2, 3 and 4
# ================================================================
def multiply(a, b):
    """Multiply two numbers."""
    return a * b


def get_current_time(timezone):
    """Return the current date and time in an IANA timezone, e.g. 'Asia/Karachi'."""
    now = datetime.now(ZoneInfo(timezone))
    return now.strftime("%Y-%m-%d %H:%M:%S %Z")


def get_city_weather(city_name):
    """Return the current weather for a city, e.g. 'Lahore'."""
    url = "https://api.weatherapi.com/v1/current.json"
    try:
        response = requests.get(url, params={"key": weather_api_key, "q": city_name.strip()}, timeout=10)
        response.raise_for_status()
        data = response.json()
        location, current = data["location"], data["current"]
    except (requests.RequestException, KeyError):
        return "Weather information not available for this city"         # Never return the raw error: its URL contains the API key
    return (f"Current weather in {location['name']}, {location['country']} "
            f"(timezone: {location['tz_id']}) as of {location['localtime']}: "
            f"{current['temp_c']}°C, {current['condition']['text']}, humidity {current['humidity']}%")


# ================================================================
# 2. Map each tool name to its function, and describe each tool to the model
# ================================================================
TOOLS = {
    "multiply": multiply,
    "get_current_time": get_current_time,
    "get_city_weather": get_city_weather,
}

TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "multiply",
            "description": "Multiply two numbers.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number"},
                    "b": {"type": "number"},
                },
                "required": ["a", "b"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Get the current date and time for a given timezone.",
            "parameters": {
                "type": "object",
                "properties": {
                    "timezone": {"type": "string", "description": "IANA timezone name, e.g. Asia/Karachi, Europe/London"},
                },
                "required": ["timezone"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_city_weather",
            "description": "Get the current weather for a city. Call this whenever the user asks what the weather is like in a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city_name": {"type": "string", "description": "The city to get the weather for, e.g. Lahore, London"},
                },
                "required": ["city_name"],
            },
        },
    },
]


# ================================================================
# 3. Run one tool call: the same as in Examples 3 and 4, without the print (the trace below prints the Action)
# ================================================================
def run_tool(tool_call):
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)
    try:
        return str(TOOLS[name](**args))
    except Exception as error:
        return f"Error: {error}"


# ================================================================
# 4. The ReAct loop: the only function that changed. It prints every pass and stops after MAX_STEPS
# ================================================================
def run_agent(messages):
    for step in range(1, MAX_STEPS + 1):                     # Guardrail: at most MAX_STEPS passes
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOL_SCHEMAS,
        )
        message = response.choices[0].message
        messages.append(message)
        print(f"\n--- Pass {step} (LLM call {step}) ---")

        thought = getattr(message, "reasoning", None)        # THINK: gpt-oss's hidden thinking, which Groq returns as message.reasoning
        if thought:
            print("Thought:", shorten(thought, 300))         # shorten() collapses the text to one line of at most 300 characters

        # No tool calls means the model is done and answered in plain text
        if not message.tool_calls:
            return message.content

        for tool_call in message.tool_calls:                 # ACT: the model asked for one or more tools
            print(f"Action: {tool_call.function.name}({tool_call.function.arguments})")
            result = run_tool(tool_call)                     # Our code runs the tool, not the model
            print("Observation:", shorten(result, 200))      # OBSERVE: the result goes back to the model as a "tool" message
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            })

    return f"I reached the step limit (MAX_STEPS = {MAX_STEPS}) before finding an answer. Please try a simpler question."   # STOP SAFELY


# ================================================================
# 5. The chat loop: the same as in Examples 3 and 4
# ================================================================
def main():
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    print("ReAct agent ready. Type 'exit' to quit.")
    while True:
        user_input = input("\nYou: ")
        if user_input.strip().lower() in ("exit", "quit"):
            break
        messages.append({"role": "user", "content": user_input})
        reply = run_agent(messages)
        print(f"\nAgent: {reply}")


if __name__ == "__main__":
    main()

ReAct agent ready. Type 'exit' to quit.

You: Which is warmer right now, Lahore or London? Then tell me the local time in the warmer city.

--- Pass 1 (LLM call 1) ---
Thought: We need to compare current weather for Lahore and London. Use get_city_weather for both. Then determine which temperature is higher. Then get current time in that city's timezone. Need timezone: Lahore is Asia/Karachi, London is Europe/London. We'll call get_city_weather for Lahore and London.
Action: get_city_weather({"city_name":"Lahore"})
Observation: Current weather in Lahore, Pakistan (timezone: Asia/Karachi) as of 2026-10-24 08:12: 21.8°C, Smoky haze, humidity 62%

--- Pass 2 (LLM call 2) ---
Thought: Now get London.
Action: get_city_weather({"city_name":"London"})
Observation: Current weather in London, United Kingdom (timezone: Europe/London) as of 2026-10-24 04:12: 7.4°C, Overcast, humidity 71%

--- Pass 3 (LLM call 3) ---
Thought: Lahore is warmer (21.8°C vs 7.4°C). Need local time in Lahore timezone A

### e. Same model, no harness
The cell below asks question 1 to the **same model** with a single plain LLM call: no system prompt, no tools and no loop.

**What to notice:** the model cannot give today's temperatures or the exact local time, because it has no way to look them up. In our runs it either said it cannot access live data, or it guessed from Lahore's *typical* October weather and even offered to "look up the latest observations", something it cannot do without a tool. Run it a few times: the wording changes, but the reply never contains live numbers.

Section d gave a correct, live answer with **the same model**. The only difference is the harness.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv('../keys/.env', override=True)
client = OpenAI(api_key=os.getenv("GROQ_API_KEY"), base_url="https://api.groq.com/openai/v1")

MODEL = "openai/gpt-oss-120b"                # The same model as in Section d

# The same question, but no harness: no system prompt, no tools, no loop. Just one LLM call
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Which is warmer right now, Lahore or London? Then tell me the local time in the warmer city."}],
)
print(response.choices[0].message.content)

**Which city is warmer right now?**  
Based on typical seasonal patterns, **Lahore (Pakistan) is usually warmer than London (UK) in late October**. Lahore sits in a subtropical climate zone, while London has a temperate maritime climate, so even on a cool day in London, Lahore’s temperatures tend to be higher.

**Local time in the warmer city (Lahore)**  
Lahore operates on **Pakistan Standard Time (PKT), which is UTC + 5** year‑round.  

- If you know the current Coordinated Universal Time (UTC), just add 5 hours to get Lahore’s local time.  
- For example, if the current UTC time were 12:00 (noon), the time in Lahore would be 17:00 (5 PM).

If you’d like an exact current temperature for each city, I can look up the latest observations from a reliable weather service—just let me know!
